# Exploracao de potencial por UF e parceiro

Notebook de exploracao para a base `epi_monetary_ufs.json` com foco em:

1. Dispersao por parceiro (importador):
   - eixo X: CAGR das exportacoes do produto pela UF para cada parceiro (%)
   - eixo Y: market share de importacoes do parceiro no SH6
   - tamanho da bolha: potencial de exportacoes da UF para o parceiro
2. Indice HHI por UF para os parceiros (importadores) em um produto SH6.

Observacao:
- a dispersao usa CAGR de exportacoes no eixo X e market share no eixo Y
- o tamanho da bolha usa potencial de exportacoes
- para o CAGR, a serie anual e estimada por BACI Brasil x share anual da UF

In [6]:
from pathlib import Path
from collections import defaultdict
import json

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px

pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

In [7]:
def resolve_existing_path(candidates: list[str]) -> Path:
    bases = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for rel in candidates:
        rel_path = Path(rel)
        if rel_path.is_absolute() and rel_path.exists():
            return rel_path
        for base in bases:
            candidate = (base / rel_path).resolve()
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'Nao foi possivel localizar nenhum dos caminhos: {candidates}')

path_detail_ufs = resolve_existing_path([
    'data/processed/epi_monetary_ufs.json',
    '../data/processed/epi_monetary_ufs.json',
])

path_demand = resolve_existing_path([
    'data/processed/demand_potential.parquet',
    '../data/processed/demand_potential.parquet',
])

path_supply = resolve_existing_path([
    'data/processed/supply_potential_ufs.parquet',
    '../data/processed/supply_potential_ufs.parquet',
])

path_shares = resolve_existing_path([
    'references/share-ufs.csv',
    '../references/share-ufs.csv',
])

path_countries = resolve_existing_path([
    'references/countries.csv',
    '../references/countries.csv',
])

path_raw_dir = resolve_existing_path([
    'data/raw',
    '../data/raw',
])

selected_uf = 'SC'
selected_sh6 = '020714'

ufs_disponiveis = sorted(
    pl.read_parquet(path_supply)
    .select(pl.col('sg_uf').cast(pl.Utf8))
    .unique()
    .get_column('sg_uf')
    .to_list()
)

sh6_disponiveis = sorted(
    pl.read_parquet(path_demand)
    .select(pl.col('sh6').cast(pl.Utf8).str.zfill(6).alias('sh6'))
    .unique()
    .get_column('sh6')
    .to_list()
)

print(f'UF selecionada: {selected_uf}')
print(f'SH6 selecionado: {selected_sh6}')
print(f'UFS disponiveis: {len(ufs_disponiveis)}')
print(f'SH6 disponiveis: {len(sh6_disponiveis)}')
print('Exemplo de UFs:', ufs_disponiveis[:10])
print('Exemplo de SH6:', sh6_disponiveis[:10])

UF selecionada: SC
SH6 selecionado: 020714
UFS disponiveis: 27
SH6 disponiveis: 5380
Exemplo de UFs: ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA']
Exemplo de SH6: ['010121', '010129', '010130', '010190', '010221', '010229', '010231', '010239', '010290', '010310']


In [8]:
def iter_json_array(path: Path, chunk_size: int = 8 * 1024 * 1024):
    decoder = json.JSONDecoder()
    with path.open('r', encoding='utf-8') as f:
        while True:
            ch = f.read(1)
            if ch == '':
                return
            if ch.isspace():
                continue
            if ch != '[':
                raise ValueError(f'JSON invalido em {path}: esperado [ no inicio')
            break

        buffer = ''
        eof = False

        while True:
            if not eof:
                chunk = f.read(chunk_size)
                if chunk == '':
                    eof = True
                else:
                    buffer += chunk

            pos = 0
            parsed_any = False

            while True:
                while pos < len(buffer) and buffer[pos] in ' \t\n\r,':
                    pos += 1

                if pos >= len(buffer):
                    break

                if buffer[pos] == ']':
                    return

                try:
                    obj, next_pos = decoder.raw_decode(buffer, pos)
                except json.JSONDecodeError:
                    break

                yield obj
                parsed_any = True
                pos = next_pos

            buffer = buffer[pos:]

            if eof:
                tail = buffer.strip()
                if tail in ('', ']'):
                    return
                if not parsed_any:
                    raise ValueError(f'Fim inesperado ao parsear {path}')


def aggregate_pairs_for_sh6(path_json: Path, sh6_target: str) -> pd.DataFrame:
    potential_map = defaultdict(float)
    current_map = defaultdict(float)

    for row in iter_json_array(path_json):
        sh6 = str(row.get('sh6')).zfill(6)
        if sh6 != sh6_target:
            continue

        uf = str(row.get('sg_uf'))
        importer = str(row.get('importer'))

        potential = float(row.get('potential_value') or 0.0)
        utilization = float(row.get('potential_utilization_ratio') or 0.0)
        current = max(potential * utilization, 0.0)

        key = (uf, importer)
        potential_map[key] += potential
        current_map[key] += current

    rows = []
    for (uf, importer), potential in potential_map.items():
        rows.append({
            'sg_uf': uf,
            'importer': importer,
            'potential_value': potential,
            'current_exports_value': current_map.get((uf, importer), 0.0),
        })

    if not rows:
        return pd.DataFrame(columns=['sg_uf', 'importer', 'potential_value', 'current_exports_value'])

    return pd.DataFrame(rows)


def load_import_market_share(path_demand_parquet: Path, sh6_target: str) -> pd.DataFrame:
    df = (
        pl.read_parquet(path_demand_parquet)
        .with_columns(pl.col('sh6').cast(pl.Utf8).str.zfill(6).alias('sh6'))
        .filter(pl.col('sh6') == sh6_target)
        .group_by('importer')
        .agg(pl.sum('projected_import_value').alias('projected_import_value'))
    )

    if df.height == 0:
        return pd.DataFrame(columns=['importer', 'projected_import_value', 'import_market_share_pct'])

    total = float(df['projected_import_value'].sum())
    if total > 0:
        df = df.with_columns((pl.col('projected_import_value') / total * 100.0).alias('import_market_share_pct'))
    else:
        df = df.with_columns(pl.lit(0.0).alias('import_market_share_pct'))

    return df.to_pandas()


def load_uf_partner_exports_cagr(
    path_raw: Path,
    path_shares_csv: Path,
    path_countries_csv: Path,
    uf_target: str,
    sh6_target: str,
    brazil_exporter_code: int = 76,
) -> pd.DataFrame:
    sh6_int = int(sh6_target)

    df_share = (
        pl.read_csv(
            path_shares_csv,
            null_values=['null'],
            schema_overrides={
                'nr_ano': pl.Int64,
                'cd_sh6': pl.Utf8,
                'sg_uf': pl.Utf8,
                'pct_participacao': pl.Float64,
            },
        )
        .with_columns(pl.col('cd_sh6').str.zfill(6).alias('cd_sh6'))
        .filter((pl.col('sg_uf') == uf_target) & (pl.col('cd_sh6') == sh6_target))
        .select(['nr_ano', 'pct_participacao'])
        .sort('nr_ano')
    )

    if df_share.height == 0:
        return pd.DataFrame(columns=[
            'importer', 'exports_start_usd', 'exports_end_usd', 'cagr_exports_pct',
            'cagr_period_start', 'cagr_period_end', 'cagr_n_years',
        ])

    max_share = float(df_share['pct_participacao'].max())
    if max_share > 1.0:
        df_share = df_share.with_columns((pl.col('pct_participacao') / 100.0).alias('pct_participacao'))

    df_bra = (
        pl.scan_csv(str(path_raw / 'BACI_HS17_Y*_V202601.csv'))
        .select([
            pl.col('t').cast(pl.Int64).alias('year'),
            pl.col('i').cast(pl.Int64).alias('exporter_code'),
            pl.col('j').cast(pl.Int64).alias('importer_code'),
            pl.col('k').cast(pl.Int64).alias('sh6'),
            pl.col('v').cast(pl.Float64).alias('value_kusd'),
        ])
        .filter(
            (pl.col('exporter_code') == brazil_exporter_code)
            & (pl.col('sh6') == sh6_int)
        )
        .group_by(['year', 'importer_code'])
        .agg(pl.sum('value_kusd').alias('value_kusd'))
        .collect()
    )

    if df_bra.height == 0:
        return pd.DataFrame(columns=[
            'importer', 'exports_start_usd', 'exports_end_usd', 'cagr_exports_pct',
            'cagr_period_start', 'cagr_period_end', 'cagr_n_years',
        ])

    df_yearly = (
        df_bra
        .join(df_share, left_on='year', right_on='nr_ano', how='left')
        .with_columns(pl.col('pct_participacao').fill_null(0.0))
        .with_columns((pl.col('value_kusd') * 1000.0 * pl.col('pct_participacao')).alias('exports_uf_usd'))
        .select(['year', 'importer_code', 'exports_uf_usd'])
    )

    countries = pl.read_csv(path_countries_csv).select([
        pl.col('country_code').cast(pl.Int64).alias('importer_code'),
        pl.col('country_iso3').cast(pl.Utf8).alias('importer'),
    ])

    yearly_pd = (
        df_yearly
        .join(countries, on='importer_code', how='left')
        .with_columns(pl.coalesce([pl.col('importer'), pl.col('importer_code').cast(pl.Utf8)]).alias('importer'))
        .select(['year', 'importer', 'exports_uf_usd'])
        .to_pandas()
    )

    if yearly_pd.empty:
        return pd.DataFrame(columns=[
            'importer', 'exports_start_usd', 'exports_end_usd', 'cagr_exports_pct',
            'cagr_period_start', 'cagr_period_end', 'cagr_n_years',
        ])

    rows = []
    for importer, g in yearly_pd.groupby('importer', dropna=False):
        g = g.sort_values('year').copy()
        g_pos = g[g['exports_uf_usd'] > 0].copy()

        if g_pos.empty:
            cagr = np.nan
            y0 = np.nan
            y1 = np.nan
            n = np.nan
            v0 = 0.0
            v1 = float(g['exports_uf_usd'].iloc[-1]) if len(g) else 0.0
        elif len(g_pos) == 1:
            cagr = np.nan
            y0 = int(g_pos['year'].iloc[0])
            y1 = int(g_pos['year'].iloc[0])
            n = 0
            v0 = float(g_pos['exports_uf_usd'].iloc[0])
            v1 = float(g_pos['exports_uf_usd'].iloc[0])
        else:
            y0 = int(g_pos['year'].iloc[0])
            y1 = int(g_pos['year'].iloc[-1])
            v0 = float(g_pos['exports_uf_usd'].iloc[0])
            v1 = float(g_pos['exports_uf_usd'].iloc[-1])
            n = y1 - y0
            cagr = ((v1 / v0) ** (1.0 / n) - 1.0) if (n > 0 and v0 > 0) else np.nan

        rows.append({
            'importer': str(importer),
            'exports_start_usd': v0,
            'exports_end_usd': v1,
            'cagr_exports_pct': cagr * 100.0 if pd.notna(cagr) else np.nan,
            'cagr_period_start': y0,
            'cagr_period_end': y1,
            'cagr_n_years': n,
        })

    return pd.DataFrame(rows).sort_values('exports_end_usd', ascending=False)


def build_scatter_dataset(df_exports_cagr: pd.DataFrame, df_import_share: pd.DataFrame) -> pd.DataFrame:
    base = df_exports_cagr.copy()
    if base.empty:
        return base

    merged = base.merge(
        df_import_share[['importer', 'import_market_share_pct']],
        on='importer',
        how='left',
    )
    merged['import_market_share_pct'] = merged['import_market_share_pct'].fillna(0.0)
    return merged.sort_values('exports_end_usd', ascending=False)


def calculate_hhi_by_uf(df_pairs: pd.DataFrame) -> pd.DataFrame:
    base = df_pairs.copy()
    if base.empty:
        return pd.DataFrame(columns=['sg_uf', 'hhi', 'n_importers', 'total_potential_value'])

    base['total_potential_value'] = base.groupby('sg_uf')['potential_value'].transform('sum')
    base = base.loc[base['total_potential_value'] > 0].copy()

    base['partner_share'] = base['potential_value'] / base['total_potential_value']

    out = (
        base.groupby('sg_uf', as_index=False)
        .agg(
            hhi=('partner_share', lambda s: float(np.square(s).sum())),
            n_importers=('importer', 'nunique'),
            total_potential_value=('potential_value', 'sum'),
        )
        .sort_values('hhi', ascending=False)
    )

    return out

In [11]:
# Reexecute esta celula sempre que trocar selected_sh6 ou selected_uf.

df_export_cagr = load_uf_partner_exports_cagr(
    path_raw=path_raw_dir,
    path_shares_csv=path_shares,
    path_countries_csv=path_countries,
    uf_target=selected_uf,
    sh6_target=selected_sh6,
 )
df_import_share = load_import_market_share(path_demand, selected_sh6)
df_scatter = build_scatter_dataset(df_export_cagr, df_import_share)

# Mantido para a secao de HHI (usa base potencial).
df_pairs_sh6 = aggregate_pairs_for_sh6(path_detail_ufs, selected_sh6)

# Potencial da UF selecionada por parceiro para usar no tamanho da bolha.
df_potential_uf = (
    df_pairs_sh6.loc[df_pairs_sh6['sg_uf'] == selected_uf, ['importer', 'potential_value']]
    .groupby('importer', as_index=False)['potential_value']
    .sum()
 )

df_scatter = df_scatter.merge(
    df_potential_uf,
    on='importer',
    how='left',
)
df_scatter['potential_value'] = df_scatter['potential_value'].fillna(0.0)

print(f'Linhas com CAGR para SH6 {selected_sh6} e UF {selected_uf}: {len(df_scatter):,}')
print(f'Parceiros com CAGR calculavel (nao nulo): {df_scatter["cagr_exports_pct"].notna().sum():,}')
print(f'Parceiros com potencial positivo: {(df_scatter["potential_value"] > 0).sum():,}')

if len(df_scatter) == 0:
    print('Sem dados para a combinacao de UF e SH6 selecionada.')
else:
    df_plot = df_scatter.dropna(subset=['cagr_exports_pct']).copy()

    if len(df_plot) == 0:
        print('Nao ha parceiros com serie suficiente para calcular CAGR nesta combinacao.')
    else:
        fig_scatter = px.scatter(
            df_plot,
            x='cagr_exports_pct',
            y='import_market_share_pct',
            size='potential_value',
            hover_name='importer',
            hover_data={
                'potential_value': ':.2f',
                'exports_start_usd': ':.2f',
                'exports_end_usd': ':.2f',
                'cagr_exports_pct': ':.3f',
                'cagr_period_start': True,
                'cagr_period_end': True,
                'cagr_n_years': True,
                'import_market_share_pct': ':.4f',
            },
            title=f'Dispersao por parceiro | UF={selected_uf} | SH6={selected_sh6}',
            labels={
                'cagr_exports_pct': 'CAGR das exportacoes da UF para o parceiro (%)',
                'import_market_share_pct': 'Market share de importacoes do parceiro no SH6 (%)',
                'potential_value': 'Potencial de exportacoes da UF para o parceiro (USD)',
            },
            template='plotly_white',
            height=650,
        )
        fig_scatter.update_traces(marker=dict(line=dict(width=0.7, color='black'), opacity=0.75))
        fig_scatter.update_xaxes(ticksuffix='%', tickformat='.2f')
        fig_scatter.show()

df_scatter.head(20)

Linhas com CAGR para SH6 020714 e UF SC: 203
Parceiros com CAGR calculavel (nao nulo): 192
Parceiros com potencial positivo: 169


,importer,exports_start_usd,exports_end_usd,cagr_exports_pct,cagr_period_start,cagr_period_end,cagr_n_years,import_market_share_pct,potential_value
0,CHN,"205,514,709.731925","285,422,787.692959",4.804025,2017,2024,7,20.098716,"371,283,827.655155"
1,JPN,"239,882,544.359405","183,896,929.275556",-3.725597,2017,2024,7,8.168485,"82,580,986.030118"
2,ARE,"78,699,656.034845","119,169,236.284970",6.106409,2017,2024,7,3.576032,"97,654,420.935309"
3,MEX,"49,602,268.337613","117,592,806.845693",13.123849,2017,2024,7,3.244257,"94,589,191.046711"
4,SAU,"122,735,834.696719","111,030,334.977931",-1.421668,2017,2024,7,3.536493,"77,850,710.181325"
5,IRQ,"22,631,564.850906","69,031,922.915415",17.271038,2017,2024,7,2.376813,"24,818,876.155249"
6,KOR,"44,512,417.474207","65,989,527.188952",5.785877,2017,2024,7,2.211062,"16,803,263.969419"
7,SGP,"39,537,469.707384","50,682,861.601547",3.611382,2017,2024,7,1.591833,"46,384,269.860246"
8,PHL,"5,096,405.263456","46,701,424.656029",37.226512,2017,2024,7,3.090864,"64,984,743.473106"
9,CHL,"18,830,116.804390","45,347,494.421053",13.377953,2017,2024,7,1.156051,"104,241,181.284682"


In [12]:
# HHI por UF para o SH6 selecionado.

df_hhi_uf = calculate_hhi_by_uf(df_pairs_sh6)

print(f'UFs comparadas no HHI: {len(df_hhi_uf):,}')

if len(df_hhi_uf) == 0:
    print('Sem dados suficientes para calcular HHI.')
else:
    fig_hhi = px.bar(
        df_hhi_uf,
        x='sg_uf',
        y='hhi',
        color='hhi',
        hover_data={
            'n_importers': True,
            'total_potential_value': ':.2f',
            'hhi': ':.6f',
        },
        title=f'HHI dos parceiros importadores por UF | SH6={selected_sh6}',
        labels={
            'sg_uf': 'UF',
            'hhi': 'Indice HHI',
            'n_importers': 'Numero de parceiros',
            'total_potential_value': 'Potencial total (USD)',
        },
        template='plotly_white',
        height=520,
    )
    fig_hhi.update_layout(coloraxis_showscale=False)
    fig_hhi.show()

df_hhi_uf

UFs comparadas no HHI: 25


,sg_uf,hhi,n_importers,total_potential_value
17,RJ,0.157076,224,"321,945.518841"
19,RR,0.117283,224,"522,688.448327"
22,SE,0.117279,224,72.659092
24,TO,0.102253,224,"413,277.735895"
12,MT,0.102014,224,"118,800,656.625404"
9,MA,0.095789,224,"132,762.360943"
4,BA,0.089566,224,"4,265,963.434528"
11,MS,0.087133,224,"317,925,846.039187"
16,PR,0.086314,224,"2,934,271,120.223603"
18,RO,0.084473,224,"1,575.778594"


## Como usar

1. Ajuste `selected_uf` e `selected_sh6` na celula de configuracao.
2. Reexecute a celula de dispersao para recalcular o CAGR de exportacoes por parceiro.
3. Na dispersao: eixo X em percentual de CAGR e tamanho da bolha por potencial de exportacoes.
4. Reexecute a celula de HHI para comparar a concentracao entre UFs no mesmo SH6.

Interpretacao rapida do HHI:
- valor mais alto: concentracao maior de destinos (menos diversificacao de parceiros)
- valor mais baixo: distribuicao mais pulverizada entre parceiros